# Homework 02 - Hybrid Images

Contact: David C. Schedl (david.schedl@fh-hagenberg.at)

Note: this is the starter pack for the **Digital Imaging / Computer Vision** homework. You do not need to use the exact same template and can start from scratch as well!

# Setup

Let's import useful libraries, first.
We'll download pairs of test images into the `hybrid_images` folder.

In [1]:
import os
import cv2 # openCV
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.express.colors import sample_colorscale
from plotly.subplots import make_subplots

!curl -LJO "https://raw.githubusercontent.com/Digital-Media/cv_data/main/hybrid_images.zip" --silent
import zipfile
with zipfile.ZipFile("hybrid_images.zip", 'r') as zip_ref:
    zip_ref.extractall(".")


# Helper function
def multi_scale_image(img:np.ndarray) -> np.ndarray:
    """ function to create an image with multiple scales of the input image, starting with half the size of the input image

    Args:
        img (np.ndarray): input image

    Returns:
        np.ndarray: image with multiple scales
    """
    small_imgs = np.zeros_like(img)
    # reduce size in steps of two to get a smaller image to view the low frequency image
    offset = (0, 0)
    for s in np.power(2,range(1, 100)):
        nd = (img.shape[0] // s, img.shape[1] // s)
        small_imgs[offset[0]:offset[0]+nd[0], offset[1]:offset[1]+nd[1], :] = cv2.resize(img, nd[::-1], interpolation=cv2.INTER_AREA)
        offset = (offset[0] + nd[0], offset[1] + nd[1])
        if np.min(nd) <= 2: # if the image gets too small, stop
            break

    return small_imgs

## Task
<a name="Task-A" id="Task-A"> </a>


The goal of this assignment is to create hybrid images. Hybrid images are static images that change in interpretation as a function of the viewing distance. The basic idea is that high frequency tends to dominate perception when viewed from a close distance but, at a distance, only the low frequency (smooth) part of the signal can be seen. By blending the high frequency portion of one image with the low-frequency portion of another, you get a hybrid image that leads to different interpretations at different distances.

Hybrid images are complex and beautiful, but also interesting from an engineering perspective.
The original idea was published in a 2006 Siggraph papber by Oliva *et al.* :
> Aude Oliva, Antonio Torralba, and Philippe G. Schyns. "Hybrid images." ACM Transactions on Graphics (TOG) 25.3 (2006): 527-532.

You can view it
[here](https://dellaert.github.io/19F-4476/misc/oliva-siggraph-2006.pdf)
or [here](https://dl.acm.org/doi/pdf/10.1145/1141911.1141919?casa_token=uAR1sUvMfWsAAAAA:bIULrLwd4g5Eb5r97h1D3W1f38dFf9-pd7-3XI1nW-VmQ46leCVpXphTLhE1saxAZGmWtz3G-yAc)

This notebook downloads a pairs of test images that you can use for creating hybrid images. You can use your own images as well.
This project is intended to familiarize you with image filtering and the frequency domain.
Think about the algorithm and implement it. Furthermore you need to find good settings for every test pair of images. How many of which frequencies do you suppress for each image to make the hybrid image look good?

**Hint(s):**
- You can use all the code that we used for image filtering and Fourier Transformation as basis.
- Be careful with data types and ranges. The high frequency image might contain negative values!


In [2]:
# Solution Task

# image 1 and 2
img1 = cv2.imread("hybrid_images/1a_dog.bmp")
img2 = cv2.imread("hybrid_images/1b_cat.bmp")
assert img1 is not None and img2 is not None, "Image not found"


def low_pass(img, sigma):
    return cv2.GaussianBlur(img.astype(np.float32), (0, 0), sigma)

def high_pass(img, sigma):
    blurred = cv2.GaussianBlur(img.astype(np.float32), (0, 0), sigma)
    return img.astype(np.float32) - blurred


sigma_low = 8
sigma_high = 5

low = low_pass(img1, sigma_low)
high = high_pass(img2, sigma_high)

img_hybrid = np.clip(low + high, 0, 255).astype(np.uint8)


# display the 2 input images and the hybrid image
fig = make_subplots(rows=1, cols=4, subplot_titles=("Image 1", "Image 2", "Hybrid Image", "Scaled Hybrid Image(s)"))
fig.add_trace(go.Image(z=img1[:,:,::-1]), row=1, col=1)
fig.add_trace(go.Image(z=img2[:,:,::-1]), row=1, col=2)
fig.add_trace(go.Image(z=img_hybrid[:,:,::-1]), row=1, col=3)
small_img = multi_scale_image(img_hybrid)
fig.add_trace(go.Image(z=small_img[:,:,::-1]), row=1, col=4)
fig.show()

## Further comments/hints:
*   You do not need to come up with super efficient implementations! It is mostly about getting into the topic.
*   Think about the problem, solve it, and evaluate your solutions on the test images (you can add pictures yourself).
*   Summarize your ideas and solutions in the report!


**Have fun!** 😸
